In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Participant Electrode Summary

This notebook uses the demographics workbook as the cohort source, applies the requested subject filters, loads each participant's Day 1 rereferenced electrode table, and summarizes unique electrode counts per brain region.

Outputs:
- participant-by-region electrode count table
- LaTeX-friendly table via `to_latex(index=False)`
- cohort demographics for the summarized participants: `age mean \u00b1 SD` and `n male`
- missing-data audit for subjects without a readable Day 1 electrode table


In [ ]:
import os
import warnings
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

# ---- Config ----
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
include_ui_subjects = True
excluded_codes = ['MS015', 'MS024', 'MS034', 'MS038']
electrode_day = 'Day1'
roi_column_candidates = ['salman_region', 'region']
target_regions = ['OFC', 'ACC', 'dmPFC', 'dlPFC', 'HPC', 'AMY']
latex_caption = 'Participants and number of electrodes per region'

participant_column = 'MSSMCode'
required_demo_columns = ['MSSMCode', 'LongVersion', 'Minerva', 'Site']
demographic_aliases: Dict[str, List[str]] = {
    'age': ['Age', 'age'],
    'gender': ['Gender', 'gender', 'Sex', 'sex'],
}
male_labels = {'m', 'male'}
female_labels = {'f', 'female'}

print('base_dir:', base_dir)
print('load_dir:', load_dir)
print('demo_file:', demo_file)
print('include_ui_subjects:', include_ui_subjects)
print('excluded_codes:', excluded_codes)
print('electrode_day:', electrode_day)
print('roi_column_candidates:', roi_column_candidates)
print('target_regions:', target_regions)
print('latex_caption:', latex_caption)


In [ ]:
def assert_required_columns(tbl: pd.DataFrame, required_cols: Sequence[str], table_name: str) -> None:
    missing = [col for col in required_cols if col not in tbl.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}')


def resolve_column(tbl: pd.DataFrame, candidates: Sequence[str], label: str) -> str:
    for col in candidates:
        if col in tbl.columns:
            return col
    raise ValueError(
        f'Could not resolve {label} column. Tried {list(candidates)} against columns {list(tbl.columns)}'
    )


def validate_participant_consistency(
    tbl: pd.DataFrame,
    participant_col: str,
    columns_to_check: Sequence[str],
    table_name: str,
) -> None:
    bad_rows = []
    for participant, grp in tbl.groupby(participant_col, dropna=False):
        conflicts = {}
        for col in columns_to_check:
            vals = grp[col].dropna().astype(str).str.strip().replace('', np.nan).dropna().unique().tolist()
            if len(vals) > 1:
                conflicts[col] = vals
        if conflicts:
            bad_rows.append({'participant': participant, 'conflicts': conflicts})
    if bad_rows:
        raise ValueError(f'{table_name} has inconsistent participant-level values: {bad_rows}')


def normalize_gender(value) -> str:
    if pd.isna(value):
        return 'missing'
    text = str(value).strip().lower()
    if text in male_labels:
        return 'male'
    if text in female_labels:
        return 'female'
    if text == '':
        return 'missing'
    return 'unexpected'


def clean_region_value(value) -> str:
    if pd.isna(value):
        return 'Unknown'
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return 'Unknown'
    return text


def load_filtered_cohort(
    demo_file: str,
    include_ui_subjects: bool,
    excluded_codes: Sequence[str],
) -> Tuple[pd.DataFrame, Dict[str, int], str, str]:
    if not os.path.isfile(demo_file):
        raise FileNotFoundError(f'Demographics workbook not found: {demo_file}')

    demo_data = pd.read_excel(demo_file, engine='openpyxl')
    assert_required_columns(demo_data, required_demo_columns, 'demographics workbook')
    age_col = resolve_column(demo_data, demographic_aliases['age'], 'age')
    gender_col = resolve_column(demo_data, demographic_aliases['gender'], 'gender')

    audit = {'n_raw_rows': int(len(demo_data))}
    subj_df = demo_data[(demo_data['LongVersion'] == 1) & (demo_data['Minerva'] == 1)].copy()
    audit['n_after_longversion_minerva'] = int(len(subj_df))

    if not include_ui_subjects:
        subj_df = subj_df[subj_df['Site'] != 'UI'].copy()
    audit['n_after_ui_filter'] = int(len(subj_df))

    subj_df[participant_column] = subj_df[participant_column].astype(str).str.strip()
    subj_df = subj_df[~subj_df[participant_column].isin(excluded_codes)].copy()
    audit['n_after_code_exclusion'] = int(len(subj_df))

    subj_df = subj_df.rename(columns={age_col: 'Age', gender_col: 'Gender'})
    optional_columns = [col for col in ['format', 'Day2'] if col in subj_df.columns]
    keep_columns = [participant_column, 'Age', 'Gender', 'Site'] + optional_columns
    subj_df = subj_df[keep_columns].copy()

    validate_participant_consistency(
        subj_df,
        participant_col=participant_column,
        columns_to_check=['Age', 'Gender', 'Site'] + optional_columns,
        table_name='filtered demographics workbook',
    )

    subj_df = (
        subj_df.sort_values(participant_column)
        .drop_duplicates(subset=[participant_column], keep='first')
        .reset_index(drop=True)
    )
    subj_df['Age'] = pd.to_numeric(subj_df['Age'], errors='coerce')
    subj_df['Gender_Normalized'] = subj_df['Gender'].apply(normalize_gender)
    audit['n_unique_subjects'] = int(subj_df[participant_column].nunique())
    return subj_df, audit, age_col, gender_col


def load_roi_electrode_counts(
    subject_ids: Sequence[str],
    base_dir: str,
    electrode_day: str,
    roi_column_candidates: Sequence[str],
    target_regions: Sequence[str],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    missing_records = []
    target_region_set = set(target_regions)

    for subject_id in subject_ids:
        elec_path = Path(base_dir) / 'projects' / 'guLab' / 'Salman' / 'EphysAnalyses' / subject_id / f'{electrode_day}_reref_elec_df'
        if not elec_path.is_file():
            missing_records.append({'MSSMCode': subject_id, 'missing_path': str(elec_path)})
            continue

        elec_df = pd.read_csv(elec_path)
        if 'label' not in elec_df.columns:
            raise ValueError(f'Subject {subject_id} electrode table is missing label column: {elec_path}')
        region_col = next((col for col in roi_column_candidates if col in elec_df.columns), None)
        if region_col is None:
            raise ValueError(
                f'Subject {subject_id} electrode table is missing ROI column. '
                f'Tried {list(roi_column_candidates)} in {elec_path}'
            )

        subject_counts = (
            elec_df.assign(
                label=elec_df['label'].astype(str).str.strip(),
                region=elec_df[region_col].apply(clean_region_value),
            )
            .loc[lambda df: (df['label'] != '') & (df['region'].isin(target_region_set))]
            .groupby('region')['label']
            .nunique()
        )

        row = {participant_column: subject_id}
        row.update({region: int(subject_counts.get(region, 0)) for region in target_regions})
        rows.append(row)

    counts_df = pd.DataFrame(rows)
    if counts_df.empty:
        raise ValueError('No readable electrode tables were found for the filtered cohort.')

    region_columns = [region for region in target_regions if region in counts_df.columns]
    counts_df = counts_df[[participant_column] + region_columns].copy()
    counts_df[region_columns] = counts_df[region_columns].fillna(0).astype(int)
    counts_df = counts_df.sort_values(participant_column).reset_index(drop=True)

    missing_df = pd.DataFrame(missing_records)
    return counts_df, missing_df


def build_output_tables(summary_df: pd.DataFrame, target_regions: Sequence[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    out = pd.DataFrame({'subj': summary_df[participant_column].astype(str)})
    for region in target_regions:
        out[region] = summary_df[region].astype(int)

    total_row = {'subj': 'Total'}
    total_row.update({region: int(out[region].sum()) for region in target_regions})
    out_with_total = pd.concat([out, pd.DataFrame([total_row])], ignore_index=True)
    return out, out_with_total


def latex_escape(value) -> str:
    text = str(value)
    replacements = {
        '&': '\\&',
        '%': '\\%',
        '$': '\\$',
        '#': '\\#',
        '_': '\\_',
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def make_manuscript_latex_table(display_df: pd.DataFrame, target_regions: Sequence[str], caption: str) -> str:
    ncols = 1 + len(target_regions)
    colspec = 'l' + ('r' * len(target_regions))
    group_start = 2
    group_end = ncols
    lines = [
        '\\begin{table}',
        '\\centering',
        f'\\caption{{{latex_escape(caption)}}}',
        '\\resizebox{\\textwidth}{!}{%',
        f'\\begin{{tabular}}{{{colspec}}}',
        '\\toprule',
        'subj & ' + f'\\multicolumn{{{len(target_regions)}}}{{c}}{{\\# of electrodes per region}} \\\\',
        f'\\cmidrule(lr){{{group_start}-{group_end}}}',
        ' & ' + ' & '.join(target_regions) + ' \\\\',
        '\\midrule',
    ]

    for _, row in display_df.iterrows():
        row_dict = row.to_dict()
        is_total = str(row_dict['subj']) == 'Total'
        cells = [latex_escape(row_dict['subj'])]
        if is_total:
            cells[0] = '\\textbf{Total}'
            cells.extend([f'\\textbf{{{int(row_dict[region])}}}' for region in target_regions])
        else:
            cells.extend([str(int(row_dict[region])) for region in target_regions])
        if is_total:
            lines.append('\\midrule')
        lines.append(' & '.join(cells) + ' \\\\')

    lines.extend([
        '\\bottomrule',
        '\\end{tabular}%',
        '}',
        '\\end{table}',
    ])
    return '\n'.join(lines)


In [ ]:
# ---- Load and validate the workbook cohort ----
cohort_df, cohort_audit, workbook_age_col, workbook_gender_col = load_filtered_cohort(
    demo_file=demo_file,
    include_ui_subjects=include_ui_subjects,
    excluded_codes=excluded_codes,
)

print('Workbook age column:', workbook_age_col)
print('Workbook gender column:', workbook_gender_col)
print('Cohort audit:', cohort_audit)
print('Filtered workbook cohort:', len(cohort_df), 'participants')

unexpected_gender_subjects = cohort_df.loc[
    cohort_df['Gender_Normalized'] == 'unexpected', [participant_column, 'Gender']
].copy()
if len(unexpected_gender_subjects) > 0:
    warnings.warn(
        'Unexpected non-missing gender labels found: '
        f"{unexpected_gender_subjects.to_dict(orient='records')}"
    )

display(cohort_df.head())


In [ ]:
# ---- Load Day 1 electrode counts for each cohort subject ----
electrode_counts_df, missing_electrode_df = load_roi_electrode_counts(
    subject_ids=cohort_df[participant_column].tolist(),
    base_dir=base_dir,
    electrode_day=electrode_day,
    roi_column_candidates=roi_column_candidates,
    target_regions=target_regions,
)

print('Subjects with readable electrode tables:', len(electrode_counts_df))
print('Subjects missing electrode tables:', len(missing_electrode_df))
display(electrode_counts_df.head())

if len(missing_electrode_df) > 0:
    display(missing_electrode_df)


In [ ]:
# ---- Merge demographics with available electrode summaries ----
summary_df = (
    cohort_df.merge(electrode_counts_df, on=participant_column, how='inner')
    .sort_values(participant_column)
    .reset_index(drop=True)
)
if summary_df.empty:
    raise ValueError('No overlap between the filtered workbook cohort and readable electrode tables.')

region_columns = [region for region in target_regions if region in electrode_counts_df.columns]
summary_df[region_columns] = summary_df[region_columns].fillna(0).astype(int)
summary_table, summary_table_with_total = build_output_tables(summary_df, region_columns)

print('Final summarized cohort:', len(summary_df), 'participants')
display(summary_table_with_total)


In [ ]:
# ---- Cohort demographics for the summarized participants ----
age_values = pd.to_numeric(summary_df['Age'], errors='coerce')
n_age = int(age_values.notna().sum())
age_mean = float(age_values.mean()) if n_age > 0 else np.nan
age_sd = float(age_values.std(ddof=1)) if n_age > 1 else np.nan
n_total = int(len(summary_df))
n_male = int((summary_df['Gender_Normalized'] == 'male').sum())

print(f'Age: {age_mean:.2f} \u00b1 {age_sd:.2f} years (n={n_age})')
print(f'Male patients: {n_male} / {n_total}')

missing_age_subjects = summary_df.loc[age_values.isna(), participant_column].tolist()
if missing_age_subjects:
    print('Subjects with missing/non-numeric age:', missing_age_subjects)

unexpected_summary_genders = summary_df.loc[
    summary_df['Gender_Normalized'] == 'unexpected', [participant_column, 'Gender']
].copy()
if len(unexpected_summary_genders) > 0:
    print('Unexpected gender labels in summarized cohort:')
    display(unexpected_summary_genders)


In [ ]:
# ---- LaTeX output and missing-data audit ----
latex_table = make_manuscript_latex_table(
    display_df=summary_table_with_total,
    target_regions=region_columns,
    caption=latex_caption,
)
print(latex_table)

if len(missing_electrode_df) > 0:
    print('\nMissing electrode table audit (excluded from final summary):')
    display(missing_electrode_df.sort_values('MSSMCode').reset_index(drop=True))
else:
    print('\nNo missing electrode tables for the filtered cohort.')
